In [1]:
import sys
from pathlib import Path
import pandas as pd

# Allow importing from src/
sys.path.append(str(Path("..").resolve()))

from src.data_prep import load_events, load_item_categories, prepare_temporal_split
from src.models import RecommenderSuite
from src.metrics import evaluate_recommendations

# 1. Load data & build leak-free temporal split (80/20 split, 7-day hit window)
events = load_events("../data/raw")
item_to_cat = load_item_categories("../data/raw")
split = prepare_temporal_split(events, train_quantile=0.80, hit_window_days=7)

train_df = split["train_df"]
gt_returning = split["gt_returning"]
gt_new = split["gt_new"]
seed_new = split["seed_new"]
get_segment = split["get_segment"]

print(f"Cutoff timestamp: {split['cutoff_dt']}")
print(f"Returning test users with hits (7-day window): {len(gt_returning):,}")
print(f"New cold-start users with post-view hits (7-day window): {len(gt_new):,}")

# 2. Fit all 3 recommenders on training data only
rec = RecommenderSuite(n_factors=64, random_state=42).fit(train_df)
catalog_size = len(rec.all_train_items)

# Build user -> training history lookup
user_history = train_df.groupby("visitorid")["itemid"].apply(list).to_dict()

# 3. Generate Top-10 recommendations for Returning Users
K = 10
pop_top_k = rec.recommend_popularity(k=K)

recs_ret_pop = {uid: pop_top_k for uid in gt_returning}
recs_ret_itemcf = {uid: rec.recommend_item_cf(user_history.get(uid, []), k=K) for uid in gt_returning}
recs_ret_svd = {uid: rec.recommend_personalized(uid, k=K) for uid in gt_returning}

results_returning = pd.DataFrame([
    {"Cohort": "Returning Users (All)", "Model": "1. Popularity Baseline", **evaluate_recommendations(recs_ret_pop, gt_returning, catalog_size, item_to_cat, k=K)},
    {"Cohort": "Returning Users (All)", "Model": "2. Item-to-Item CF", **evaluate_recommendations(recs_ret_itemcf, gt_returning, catalog_size, item_to_cat, k=K)},
    {"Cohort": "Returning Users (All)", "Model": "3. Personalized SVD", **evaluate_recommendations(recs_ret_svd, gt_returning, catalog_size, item_to_cat, k=K)},
])

print("\n=== RETURNING USERS: OVERALL MODEL COMPARISON (K=10) ===")
display(results_returning)

Cutoff timestamp: 2015-08-18 04:23:31.966000+00:00
Returning test users with hits (7-day window): 327
New cold-start users with post-view hits (7-day window): 2,494

=== RETURNING USERS: OVERALL MODEL COMPARISON (K=10) ===


,Cohort,Model,Users_Evaluated,Precision@10,Recall@10,NDCG@10,MAP@10,MRR@10,Catalog_Coverage,Category_Diversity
0,Returning Users (All),1. Popularity Baseline,327,0.001835,0.015291,0.015983,0.015291,0.018349,0.000047,0.900000
1,Returning Users (All),2. Item-to-Item CF,327,0.006422,0.024083,0.019059,0.014173,0.023096,0.010591,0.704281
2,Returning Users (All),3. Personalized SVD,327,0.003670,0.019495,0.012357,0.008758,0.010992,0.003090,0.758104


In [2]:
segment_rows = []

# A. Slice Returning Users by Prior Activity Level (1 event vs 2-4 vs >=5)
segments = ["Cold-Start (1 event)", "Light Browser (2-4 events)", "Heavy Browser (>=5 events)"]

for seg in segments:
    seg_gt = {uid: items for uid, items in gt_returning.items() if get_segment(uid) == seg}
    if not seg_gt:
        continue
    for model_name, recs_dict in [
        ("1. Popularity Baseline", recs_ret_pop),
        ("2. Item-to-Item CF", recs_ret_itemcf),
        ("3. Personalized SVD", recs_ret_svd),
    ]:
        metrics = evaluate_recommendations(recs_dict, seg_gt, catalog_size, item_to_cat, k=K)
        segment_rows.append({"Segment": seg, "Model": model_name, **metrics})

# B. Evaluate New Users (Zero prior train history; anchored on their 1st test view)
recs_new_pop = {uid: pop_top_k for uid in gt_new}
recs_new_itemcf = {uid: rec.recommend_item_cf([seed_new[uid]], k=K) for uid in gt_new}
recs_new_svd = {uid: rec.recommend_personalized(uid, k=K) for uid in gt_new}  # falls back to popularity

for model_name, recs_dict in [
    ("1. Popularity Baseline", recs_new_pop),
    ("2. Item-to-Item CF (1st-View Session)", recs_new_itemcf),
    ("3. Personalized SVD (Cold Fallback)", recs_new_svd),
]:
    metrics = evaluate_recommendations(recs_dict, gt_new, catalog_size, item_to_cat, k=K)
    segment_rows.append({"Segment": "New Users (0 train events)", "Model": model_name, **metrics})

segment_df = pd.DataFrame(segment_rows)
print("=== SEGMENT-BY-SEGMENT BENCHMARK RESULTS ===")
display(segment_df)

# Save combined results for the Streamlit dashboard
all_results = pd.concat([
    results_returning.rename(columns={"Cohort": "Segment"}),
    segment_df
], ignore_index=True)
all_results.to_csv("../data/processed/evaluation_results.csv", index=False)
print("Saved -> data/processed/evaluation_results.csv")

=== SEGMENT-BY-SEGMENT BENCHMARK RESULTS ===


,Segment,Model,Users_Evaluated,Precision@10,Recall@10,NDCG@10,MAP@10,MRR@10,Catalog_Coverage,Category_Diversity
0,Cold-Start (1 event),1. Popularity Baseline,98,0.002041,0.015306,0.016461,0.015306,0.020408,0.000047,0.900000
1,Cold-Start (1 event),2. Item-to-Item CF,98,0.002041,0.015306,0.009891,0.006803,0.011905,0.003640,0.729592
2,Cold-Start (1 event),3. Personalized SVD,98,0.002041,0.020408,0.014152,0.012245,0.012245,0.001775,0.779592
3,Light Browser (2-4 events),1. Popularity Baseline,81,0.004938,0.043210,0.044607,0.043210,0.049383,0.000047,0.900000
4,Light Browser (2-4 events),2. Item-to-Item CF,81,0.007407,0.030864,0.018462,0.009906,0.021223,0.003250,0.658025
5,Light Browser (2-4 events),3. Personalized SVD,81,0.004938,0.018519,0.009494,0.004081,0.008230,0.001620,0.759259
6,Heavy Browser (>=5 events),1. Popularity Baseline,148,0.000000,0.000000,0.000000,0.000000,0.000000,0.000047,0.900000
7,Heavy Browser (>=5 events),2. Item-to-Item CF,148,0.008784,0.026182,0.025455,0.021388,0.031532,0.005143,0.712838
8,Heavy Browser (>=5 events),3. Personalized SVD,148,0.004054,0.019426,0.012736,0.009009,0.011674,0.002095,0.743243
9,New Users (0 train events),1. Popularity Baseline,2494,0.000642,0.003719,0.003909,0.003362,0.005528,0.000047,0.900000


Saved -> data/processed/evaluation_results.csv
